## Paper numbers reproduced by this script
Paper: "The Limits of Query Monitoring for Distributed Model Extraction".
Run on Kaggle (see "Kaggle:" line below for hardware/time). Seeds are fixed in the configuration cell;
re-runs vary by a few points (the paper reports run-to-run spread up to ~0.07-0.14 for weekly detection).

| Paper location | Quantity | Value in paper | Where computed here |
|---|---|---|---|
| App. P (Step 21 rows) | locked predictions P21.1-P21.7 | Fail (0.03 vs 0.03); Pass (1.00; random-group null); Pass (uninformative: both ≈ 0); Fail (0.035 vs 0.447); Pass (0.92; 0%); Pass (0.95 to 0.12); Pass (0.88 vs 0.70) | final cell "Predictions, figure, VERDICT REPORT", prints "P21.1/2" ... "P21.7" |
| §5 'Attacks on the monitor'; App. D.6 'Choosing who gets flagged' | sacrificial scheduling (30% most distinctive queries on five accounts): detection on C; other colluders in the flagged group | 0.92; none of the other 45 colluders | Part X, `SCHED["sacrificial"]` (`paired`, `recall_nonsac`); print "[X] sacrificial" |
| §5 'Attacks on the monitor'; App. D.6 'Teaching the monitor that the campaign is normal' | campaign ramped through the eight calibration weeks vs abrupt: deployed two-scale detection | from 0.95 to 0.12 | Part T, `TEMP["abrupt"]` / `TEMP["ramped"]`; print "[T] deployed two-scale paired detection" |
| App. D.2 'Its edge is calibration, not ranking' | oracle TPR at 10−3 false alarms, MetaMathQA: two-scale; global stream; MMD | 0.88; 0.70; 0.39 | Part R, `LOWFPR[("metamath_unique", d)][1e-3]`; print "[R] metamath_unique ..." |
| App. F 'Scale' paragraph; App. D.5 'Why this is not yet a production result' (sparse scans did not help) | sparse scans (truncated power iteration up to 100 iterations x 20 restarts; greedy forward scan) vs an oracle told the colluders | caught 0–4% of campaigns at every colluder fraction from 1% to 28%; oracle caught all of them | Part S, `SPARSE_ROWS` (`tp_*`, `greedy_*`, `oracle`); print "[S] weekly/pooled K=..." |
| App. F 'Statistics that failed' | mixing a fraction f of natural prompts (lam here): oracle two-scale detection; MMD2 where it crosses 0.5; benign top groups' distance | 0.81 (f = 0) to 0.00 (f = 0.75); about 0.035; median 0.145 | Part B, `BOUND` (`tpr`, `eps`, `eps_star`) and `GEPS`; print "[B] weekly: lam=..." |

Not reported in the paper: balanced scheduling and random-schedule precision/recall; metamath_mixed sparse rows; per-statistic compute times; pooled boundary rows; TPR at FPR 10^-1 and 10^-2, and metamath_mixed low-FPR rows; step21_figure.png.

# Step 21 — Sparse-support detection, an empirical detection boundary, adversarial scheduling, temporal camouflage, and low-FPR ROC
**Kaggle:** GPU + Internet + `HF_TOKEN` → **Run All** (≈80–100 min; reuses earlier caches). `QUICK=True` first.

* **S — sparse-support detection (oracle → sparse scan → spectral).** On the whitened account rows Y, the sparse eigenvalue max over ‖z‖₀ ≤ k of zᵀ(YYᵀ/r)z is approximated by **truncated power iteration** (TPower; k ∈ {K, 2K}, iterations × restarts ∈ {5×1, 20×5, 100×20}) and by **greedy forward scan**. Compared with the spectral λmax, the two-scale test, and an **oracle told the colluders** (the colluder union's S̄ against random K-account unions in null windows). Campaign fixed at N = 1,640 queries over K ∈ {10, 25, 50}; weekly windows (K/A ≈ 5–35%) and pooled populations of A = 1,000 account-weeks (K/A = 1–5%). Power at oracle-matched 1% FPR against measured compute.
* **B — empirical detection boundary.** Campaigns interpolate toward benign traffic: P_λ = (1−λ)·MetaMathQA + λ·natural prompts, within every account, λ ∈ {0, 0.25, 0.5, 0.75, 0.9}. The law gap ε is measured as MMD² between campaign and benign reference queries; the boundary ε*(A) is where oracle TPR crosses 0.5, compared with the distribution of ε for the most distinctive benign group in null windows.
* **X — cross-account adversarial scheduling.** The same campaign assigned to accounts by `random`, `balanced` (each account receives the same mix of most- and least-distinctive prompts) or `sacrificial` (the 30% most distinctive prompts on 5 accounts) schedules; detection and which colluders get flagged.
* **T — temporal camouflage.** The campaign ramps up over the 8 calibration weeks before a test week (volume rising linearly to full), so it contaminates the lagged reference and the calibration windows; compared with an abrupt campaign.
* **R — low-FPR ROC.** 100 extra null windows per test week (≈ 3,300 in total) give oracle TPR at FPR 10⁻¹, 10⁻² and 10⁻³ (10⁻⁴ is not estimable).

| ID | Locked prediction |
|---|---|
| **P21.1** | At K/A ≤ 5% (pooled A = 1,000), the best sparse scan has oracle TPR at 1% ≥ spectral + 0.2, at ≤ 100× spectral compute |
| **P21.2** | The oracle told the colluders detects at K/A = 1% with TPR ≥ 0.8 (a large statistical–computational gap) |
| **P21.3** | Where the spectral test already works (weekly, K/A ≥ 10%), sparse scans add ≤ 0.1 |
| **P21.4** | The boundary ε*(A) rises with A, and lies within a factor of 2 of the 99th percentile of the most distinctive benign group's ε |
| **P21.5** | Sacrificial scheduling: two-scale detection stays ≥ 0.6, but the flagged group contains ≤ 10% of the non-sacrificial colluders |
| **P21.6** | Temporal camouflage cuts deployed two-scale detection by ≥ 0.3 relative to an abrupt campaign |
| **P21.7** | Weekly MetaMathQA campaigns: two-scale oracle TPR at FPR 10⁻³ ≥ 0.5, and ≥ global-stream TPR |

In [ ]:
# ===== Cell 1: configuration =====
import math, json, time, os, gc, re, pickle
import numpy as np
QUICK = False
SYNTHETIC_TEST = False
SEED = 9091
OUTDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
try:
    import torch
    HAS_TORCH = True; DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception:
    HAS_TORCH, DEV = False, None
USE_GPU = HAS_TORCH and DEV.type == "cuda"
if not USE_GPU and not SYNTHETIC_TEST:
    raise RuntimeError("No GPU visible. Kaggle: Settings -> Accelerator -> GPU, then restart.")
CHUNK = 32768 if USE_GPU else 8192
H_COARSE, H_FINE, M_MODES, ALPHA, FLOOR = 4.0, 0.5, 300, 0.01, 1e-4
VOL_CAP, MIN_PER_ACCT, POOL_FRAC, SUB_FRAC = 500, 10, 0.2, 0.7
N_TOPICS, N_REF_BATCH, REF_BATCH, MMD_WIN, MINCELL = 100, 20, 500, 2000, 20
DETS = ["twoscale", "qc", "qcount", "mmd", "gmmd", "glob", "spec", "fraudar"]
SRC_MAIN = ["metamath_unique", "metamath_mixed", "metamath_followup", "gsm8k_human", "random"]
EVALS = [("eval_gsm8k", 3, 300), ("eval_gsm8k", 10, 100), ("eval_mmlu", 3, 300), ("eval_mmlu", 10, 100)]
SRC_EXTRA = ["wild_cluster"]
CAMO_FRAC = 0.2
if QUICK:
    WK = dict(KS=[20], CAL_N=4, LAGB=2, N_TEST=3, NULL_PER=12, REPS=2)
    POP = dict(AS=[40, 70], K=6, N_FIX=200, STEP=4, FIRST=12, N_CAL=12, N_TEST=6, REPS=3)
    P21 = dict(KS=[4, 8], N_FIX=200, A_POOL=80, NULL_POOL=30, ATT_POOL=6, LAMBDAS=[0.0, 0.5, 0.9], T_WEEKS=2, T_CAL=6, T_REPS=3, R_EXTRA=10)
    EVALS = [("eval_gsm8k", 3, 40), ("eval_mmlu", 3, 40)]
    MMD_WIN, N_REF_BATCH, REF_BATCH = 400, 5, 200
else:
    WK = dict(KS=[50], CAL_N=8, LAGB=4, N_TEST=24, NULL_PER=40, REPS=4)
    POP = dict(AS=[250, 500, 1000], K=50, N_FIX=1640, STEP=4, FIRST=20, N_CAL=40, N_TEST=20, REPS=10)
    P21 = dict(KS=[10, 25, 50], N_FIX=1640, A_POOL=1000, NULL_POOL=200, ATT_POOL=30, LAMBDAS=[0.0, 0.25, 0.5, 0.75, 0.9], T_WEEKS=6, T_CAL=20, T_REPS=10, R_EXTRA=100)
CACHE11 = f"{OUTDIR}/step11_wildchat_E384_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
CACHE13 = f"{OUTDIR}/step13_distill_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
CACHE14 = f"{OUTDIR}/step14_meta_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
CACHE15 = f"{OUTDIR}/step15_extra_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"
N_CONV = 30000 if QUICK else None; DIMS = [384]
print(f"backend {'GPU ' + torch.cuda.get_device_name(0) if USE_GPU else 'CPU'} | QUICK={QUICK}")
CACHE18 = f"{OUTDIR}/step18_evals_{'quick' if QUICK else 'full'}{'_synth' if SYNTHETIC_TEST else ''}.npz"

## Part 0 — caches

In [ ]:
# ===== Cell 2: full 384-d MiniLM embeddings of English user turns (cached as float16) =====
if os.path.exists(CACHE11):
    _c = np.load(CACHE11); E16, ACCT, WEEK = _c["E16"], _c["ACCT"], _c["WEEK"]; print(f"loaded cache: {E16.shape}")
else:
    t0 = time.time()
    if SYNTHETIC_TEST:
        rng = np.random.default_rng(0); n_ip, n_rows = 3000, 60000
        ACCT = rng.integers(0, n_ip, n_rows).astype(np.int32); WEEK = rng.integers(0, 30, n_rows).astype(np.int16)
        topic = rng.integers(0, 40, n_ip)[ACCT]; base = rng.standard_normal((40, 384))
        E = (base[topic] * 0.8 + rng.standard_normal((n_rows, 384)) * 0.6).astype(np.float32)
    else:
        from datasets import load_dataset
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = os.environ.get("HF_TOKEN")
        split = "train" if N_CONV is None else f"train[:{N_CONV}]"
        ds = load_dataset("allenai/WildChat-1M", split=split, token=tok).select_columns(["conversation", "hashed_ip", "timestamp", "language"])
        texts, ips, tss = [], [], []
        for batch in ds.iter(batch_size=2000):
            for conv, ip, ts, lang in zip(batch["conversation"], batch["hashed_ip"], batch["timestamp"], batch["language"]):
                if lang != "English" or ip is None or ts is None: continue
                for turn in conv:
                    if turn.get("role") == "user" and turn.get("content"):
                        texts.append(turn["content"][:1000]); ips.append(ip); tss.append(ts)
        print(f"{len(texts)} English user turns from {len(set(ips))} IPs ({time.time()-t0:.0f}s)")
        _, ACCT = np.unique(np.array(ips), return_inverse=True); ACCT = ACCT.astype(np.int32)
        days = np.array([np.datetime64(t.replace(tzinfo=None) if hasattr(t, "replace") else t, "D") for t in tss])
        WEEK = ((days - days.min()).astype(np.int64) // 7).astype(np.int16); del ds; gc.collect()
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        E = enc.encode(texts, batch_size=512, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
        del texts; gc.collect()
    E16 = E.astype(np.float16); del E; gc.collect()
    np.savez(CACHE11, E16=E16, ACCT=ACCT, WEEK=WEEK); print(f"cached ({time.time()-t0:.0f}s)")
n_w = int(WEEK.max()) + 1
# PCA basis on a 100k fit sample (nested: the first d components of the same basis for every d)
rng = np.random.default_rng(SEED); fit = E16[rng.choice(len(E16), min(100000, len(E16)), replace=False)].astype(np.float32)
MU_E = fit.mean(0); ew, U = np.linalg.eigh(np.cov(fit - MU_E, rowvar=False)); EW, UB = ew[::-1], U[:, ::-1].copy()
for d in DIMS: print(f"d={d:3d}: explains {EW[:d].sum()/EW.sum():.3f} of embedding variance")
def make_Z(d):
    W_ = UB[:, :d] / math.sqrt(EW[:d].sum() / 4.0); Z = np.empty((len(E16), d), dtype=np.float32)
    for s0 in range(0, len(E16), 200000): Z[s0:s0 + 200000] = (E16[s0:s0 + 200000].astype(np.float32) - MU_E) @ W_
    return Z

In [ ]:
# ===== Cell 3: conversation ids, turn indices and attacker-pool text pairs (same order as the cache), MetaMathQA pairs =====
import pickle
rng = np.random.default_rng(SEED + 1); n_acct = int(ACCT.max()) + 1; is_pool = rng.random(n_acct) < POOL_FRAC    # identical to Steps 12-13
PAIRS_PKL = CACHE14.replace(".npz", "_pairs.pkl")
if os.path.exists(CACHE14) and os.path.exists(PAIRS_PKL):
    _c = np.load(CACHE14); CONV, TURN, D_MMU = _c["CONV"], _c["TURN"], _c["D_MMU"]
    with open(PAIRS_PKL, "rb") as fh: POOL_TXT, MM_PAIRS = pickle.load(fh)
    print(f"loaded metadata cache: {len(CONV)} turns, {len(POOL_TXT)} pool pairs, {len(MM_PAIRS)} MetaMathQA pairs")
else:
    t0 = time.time()
    if SYNTHETIC_TEST:
        r = np.random.default_rng(5); CONV = np.cumsum(r.random(len(E16)) < 0.4).astype(np.int32); TURN = np.ones(len(E16), np.int16)
        for i in range(1, len(E16)):
            if CONV[i] == CONV[i - 1]: TURN[i] = TURN[i - 1] + 1
        POOL_TXT = {int(i): (f"user prompt {i}", f"assistant reply {i} " * 5) for i in np.flatnonzero(is_pool[ACCT])}
        MM_PAIRS = [(f"What is {a} plus {a+1}?", f"{a} + {a+1} = {2*a+1}. The answer is: {2*a+1}") for a in range(1500)]
        D_MMU = (np.random.default_rng(3).standard_normal(384) * 2 + np.random.default_rng(4).standard_normal((1500, 384)) * 0.7).astype(np.float16)
    else:
        from datasets import load_dataset
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = os.environ.get("HF_TOKEN")
        split = "train" if N_CONV is None else f"train[:{N_CONV}]"
        ds = load_dataset("allenai/WildChat-1M", split=split, token=tok).select_columns(["conversation", "hashed_ip", "timestamp", "language"])
        ips, conv_id, turn_ix, users, replies = [], [], [], [], []; cid = 0
        for batch in ds.iter(batch_size=2000):
            for conv, ip, ts, lang in zip(batch["conversation"], batch["hashed_ip"], batch["timestamp"], batch["language"]):
                if lang != "English" or ip is None or ts is None: continue
                t = 0
                for j, turn in enumerate(conv):
                    if turn.get("role") == "user" and turn.get("content"):
                        t += 1; nxt = next((c.get("content") or "" for c in conv[j + 1:] if c.get("role") == "assistant"), "")
                        ips.append(ip); conv_id.append(cid); turn_ix.append(t); users.append(turn["content"][:1000]); replies.append(nxt[:1500])
                cid += 1
        _, ACCT2 = np.unique(np.array(ips), return_inverse=True)
        assert len(ACCT2) == len(ACCT) and np.array_equal(ACCT2.astype(np.int32), ACCT), "re-read order does not match the Step 11 cache"
        CONV, TURN = np.array(conv_id, np.int32), np.array(turn_ix, np.int16)
        POOL_TXT = {int(i): (users[i], replies[i]) for i in np.flatnonzero(is_pool[ACCT])}
        del users, replies, ds; gc.collect()
        print(f"re-read WildChat: order verified, {len(CONV)} turns in {int(CONV.max())+1} conversations, {len(POOL_TXT)} pool pairs ({time.time()-t0:.0f}s)")
        mm = load_dataset("meta-math/MetaMathQA", split="train", token=tok).select_columns(["query", "response", "original_question"])
        groups = {}
        for q, a, o in zip(mm["query"], mm["response"], mm["original_question"]):
            if q and a and o: groups.setdefault(o, []).append((q.strip()[:1000], a.strip()[:2000]))
        rr = np.random.default_rng(SEED); origs = list(groups); rr.shuffle(origs)
        MM_PAIRS = [groups[o][int(rr.integers(len(groups[o])))] for o in origs]          # one (query, response) per source problem
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        D_MMU = enc.encode([q for q, _ in MM_PAIRS], batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        print(f"MetaMathQA: {len(MM_PAIRS)} (query, response) pairs, one per source problem ({time.time()-t0:.0f}s)")
    np.savez(CACHE14, CONV=CONV, TURN=TURN, D_MMU=D_MMU)
    with open(PAIRS_PKL, "wb") as fh: pickle.dump((POOL_TXT, MM_PAIRS), fh)
DIST = {"metamath_unique": D_MMU}
print(f"follow-up turns (turn >= 2) among all turns: {np.mean(TURN >= 2):.3f}")

In [ ]:
# ===== Cell 3: MetaMathQA grouped by source problem (unique vs rephrased) + Magicoder, MiniLM, cached =====
if os.path.exists(CACHE13):
    _c = np.load(CACHE13); DIST13 = {k[2:]: _c[k] for k in _c.files if k.startswith("D_")}; MM_GROUP = _c["MM_GROUP"]; print({k: v.shape for k, v in DIST13.items()})
else:
    t0 = time.time(); DIST13 = {}
    if SYNTHETIC_TEST:
        r = np.random.default_rng(3); c = r.standard_normal(384) * 2
        base = c + r.standard_normal((1500, 384)) * 0.7
        DIST13["metamath_unique"] = base.astype(np.float16)
        reph = np.repeat(base[:600], 4, axis=0) + r.standard_normal((2400, 384)) * 0.05; DIST13["metamath_rephrase"] = reph.astype(np.float16)
        MM_GROUP = np.repeat(np.arange(600), 4)
        DIST13["magicoder"] = (r.standard_normal(384) * 2 + r.standard_normal((2000, 384)) * 0.7).astype(np.float16)
    else:
        from datasets import load_dataset
        tok = None
        try:
            from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = os.environ.get("HF_TOKEN")
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        emb = lambda T: enc.encode(T, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        mm = load_dataset("meta-math/MetaMathQA", split="train", token=tok).select_columns(["query", "original_question"])
        qs, oq = mm["query"], mm["original_question"]; groups = {}
        for q, o in zip(qs, oq):
            if q and o: groups.setdefault(o, set()).add(q.strip()[:1000])
        rng = np.random.default_rng(SEED); origs = list(groups); rng.shuffle(origs)
        uniq = [sorted(groups[o])[int(rng.integers(len(groups[o])))] for o in origs]
        multi = [o for o in origs if len(groups[o]) >= 4][:6000]
        reph, gid = [], []
        for j, o in enumerate(multi):
            for q in rng.permutation(sorted(groups[o]))[:4]: reph.append(q); gid.append(j)
        DIST13["metamath_unique"] = emb(uniq); DIST13["metamath_rephrase"] = emb(reph); MM_GROUP = np.array(gid)
        print(f"MetaMathQA: {len(uniq)} source problems (unique set), {len(multi)} problems x 4 variants (rephrase set) ({time.time()-t0:.0f}s)")
        mg = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train", token=tok)
        idx = np.random.default_rng(SEED).choice(len(mg), min(25000, len(mg)), replace=False)
        DIST13["magicoder"] = emb(list(dict.fromkeys(str(mg[int(i)]["problem"])[:1000] for i in idx)))
        print(f"Magicoder: {len(DIST13['magicoder'])} prompts ({time.time()-t0:.0f}s)")
    np.savez(CACHE13, MM_GROUP=MM_GROUP, **{f"D_{k}": v for k, v in DIST13.items()})

In [ ]:
# ===== Cell 4: extra sources — human GSM8K questions, templated MetaMathQA, natural-likeness scores (cached) =====
if os.path.exists(CACHE15):
    _c = np.load(CACHE15); D_GSM, D_TMPL, NAT = _c["D_GSM"], _c["D_TMPL"], _c["NAT"]
else:
    t0 = time.time()
    if SYNTHETIC_TEST:
        r = np.random.default_rng(11); D_GSM = (D_MMU[:1000].astype(np.float32) + r.standard_normal((1000, 384)) * 0.3).astype(np.float16)
        D_TMPL = (D_MMU.astype(np.float32) * 0.7 + 0.3).astype(np.float16)
    else:
        from datasets import load_dataset
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        emb = lambda T: enc.encode(T, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        g = load_dataset("openai/gsm8k", "main", split="train")
        D_GSM = emb([q[:1000] for q in g["question"]])                                    # human-written math word problems
        TEMPLATE = "You are a careful math tutor. Solve the following problem step by step and state the final answer.\n\n"
        D_TMPL = emb([TEMPLATE + q for q, _ in MM_PAIRS])                                   # same problems, fixed template prefix
    # natural-likeness of each MetaMathQA prompt: mean cosine similarity to its 10 nearest benign WildChat turns
    ben = np.flatnonzero(~is_pool[ACCT]); smp = np.random.default_rng(3).choice(ben, min(50000, len(ben)), replace=False)
    B_ = E16[smp].astype(np.float32); Q_ = D_MMU.astype(np.float32); NAT = np.empty(len(Q_), np.float32)
    for s0 in range(0, len(Q_), 2048):
        if USE_GPU:
            sims = torch.as_tensor(Q_[s0:s0 + 2048], device=DEV) @ torch.as_tensor(B_, device=DEV).T
            NAT[s0:s0 + 2048] = sims.topk(10, dim=1).values.mean(1).cpu().numpy()
        else:
            sims = Q_[s0:s0 + 2048] @ B_.T; NAT[s0:s0 + 2048] = np.sort(sims, 1)[:, -10:].mean(1)
    np.savez(CACHE15, D_GSM=D_GSM, D_TMPL=D_TMPL, NAT=NAT); print(f"extra sources built ({time.time()-t0:.0f}s)")
CAMO = np.flatnonzero(NAT >= np.quantile(NAT, 1 - CAMO_FRAC))
DIST = {"metamath_unique": D_MMU, "metamath_rephrase": DIST13["metamath_rephrase"], "gsm8k_human": D_GSM, "metamath_template": D_TMPL}
print(f"sources: " + ", ".join(f"{k}:{len(v)}" for k, v in DIST.items()) + f" | camouflage pool {len(CAMO)} (natural-likeness >= {np.quantile(NAT, 1 - CAMO_FRAC):.3f}; all MetaMathQA median {np.median(NAT):.3f})")
# ---- benchmark sweeps for the benign evaluation-pipeline test (GSM8K test questions, MMLU questions)
if os.path.exists(CACHE18):
    _c = np.load(CACHE18); D_EG, D_EM = _c["D_EG"], _c["D_EM"]
else:
    if SYNTHETIC_TEST:
        r = np.random.default_rng(21); D_EG = (D_GSM[:300].astype(np.float32) + r.standard_normal((300, 384)) * 0.2).astype(np.float16)
        D_EM = (r.standard_normal(384) * 2 + r.standard_normal((800, 384)) * 0.6).astype(np.float16)
    else:
        from datasets import load_dataset
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            os.system("pip install -q sentence-transformers"); from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda"); enc.max_seq_length = 256
        emb = lambda T: enc.encode(T, batch_size=512, convert_to_numpy=True, normalize_embeddings=True).astype(np.float16)
        D_EG = emb([q[:1000] for q in load_dataset("openai/gsm8k", "main", split="test")["question"]])
        mm = load_dataset("cais/mmlu", "all", split="test")
        D_EM = emb([(q + "\n" + "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(ch)))[:1000] for q, ch in zip(mm["question"], mm["choices"])])
    np.savez(CACHE18, D_EG=D_EG, D_EM=D_EM)
DIST["eval_gsm8k"], DIST["eval_mmlu"] = D_EG, D_EM
print(f"evaluation sweeps: GSM8K test {len(D_EG)}, MMLU {len(D_EM)}")

## Part 1 — projection, topics, statistics

In [ ]:
# ===== Cell 4: projection, topics, natural source cluster with farthest-point order =====
from sklearn.cluster import MiniBatchKMeans
rng = np.random.default_rng(SEED + 1); n_acct = int(ACCT.max()) + 1; is_pool = rng.random(n_acct) < POOL_FRAC; pool_idx = np.flatnonzero(is_pool[ACCT])
Z = make_Z(384); W384 = UB[:, :384] / math.sqrt(EW[:384].sum() / 4.0); EN = E16
DZ = {k: ((v.astype(np.float32) - MU_E) @ W384).astype(np.float32) for k, v in DIST.items()}
t0 = time.time(); ben = np.flatnonzero(~is_pool[ACCT])
km = MiniBatchKMeans(n_clusters=N_TOPICS, random_state=SEED, batch_size=4096, n_init=3).fit(Z[np.random.default_rng(1).choice(ben, min(200000, len(ben)), replace=False)])
TOPIC = km.predict(Z).astype(np.int32)
CL10 = MiniBatchKMeans(n_clusters=10, random_state=SEED + 1, batch_size=4096, n_init=3).fit(Z[pool_idx]).labels_
WILD = pool_idx[CL10 == int(np.argmax(np.bincount(CL10)))]
def fps(X, N, seed=0):
    N = min(N, len(X)); r = np.random.default_rng(seed); i0 = int(r.integers(len(X)))
    if USE_GPU:
        Xt = torch.as_tensor(X, dtype=torch.float32, device=DEV); d2 = ((Xt - Xt[i0]) ** 2).sum(1); sel = torch.empty(N, dtype=torch.long, device=DEV); sel[0] = i0
        for t in range(1, N): i = torch.argmax(d2); sel[t] = i; d2 = torch.minimum(d2, ((Xt - Xt[i]) ** 2).sum(1))
        return sel.cpu().numpy()
    sel = [i0]; d2 = ((X - X[i0]) ** 2).sum(1)
    for _ in range(N - 1): i = int(np.argmax(d2)); sel.append(i); d2 = np.minimum(d2, ((X - X[i]) ** 2).sum(1))
    return np.array(sel)
WILD_FPS = WILD[fps(Z[WILD], min(len(WILD), 20000), seed=SEED)]
_sub = EN[np.random.default_rng(2).choice(len(EN), 2000, replace=False)].astype(np.float32)
_d2 = ((_sub[:1000, None, :] - _sub[None, 1000:, :]) ** 2).sum(-1); MMD_BW = float(np.sqrt(np.median(_d2) / 2)); del _d2
print(f"natural source cluster {len(WILD)} prompts (farthest-point order {len(WILD_FPS)}); sources: " + ", ".join(f"{k}:{len(v)}" for k, v in DIST.items()) + f" ({time.time()-t0:.0f}s)")

In [ ]:
# ===== Cell 5: augmented real whitening, statistics, MMD, Fraudar =====
def make_bank_aug(Zref, h, seed):
    r = np.random.default_rng(seed); modes = r.standard_normal((M_MODES, Zref.shape[1])) / h / (2 * np.pi); m2 = 2 * M_MODES
    if USE_GPU:
        Mt = torch.as_tensor(modes, dtype=torch.float64, device=DEV); G = torch.zeros((m2, m2), dtype=torch.float64, device=DEV); mu = torch.zeros(m2, dtype=torch.float64, device=DEV)
        for s0 in range(0, len(Zref), 16384):
            th = (2 * math.pi) * (torch.as_tensor(Zref[s0:s0 + 16384], dtype=torch.float64, device=DEV) @ Mt.T)
            X = torch.cat([torch.cos(th), torch.sin(th)], 1); G += X.T @ X; mu += X.sum(0)
        G, mu = G.cpu().numpy(), mu.cpu().numpy()
    else:
        G = np.zeros((m2, m2)); mu = np.zeros(m2)
        for s0 in range(0, len(Zref), 4096):
            th = 2 * np.pi * (Zref[s0:s0 + 4096] @ modes.T); X = np.hstack([np.cos(th), np.sin(th)]); G += X.T @ X; mu += X.sum(0)
    G /= len(Zref); mu /= len(Zref); Sig = G - np.outer(mu, mu); ew, U = np.linalg.eigh((Sig + Sig.T) / 2); keep = ew > FLOOR * ew.max()
    B = dict(modes=modes, mu=mu, W=U[:, keep] / np.sqrt(ew[keep]), r=int(keep.sum()))
    if USE_GPU:
        B["MT"] = torch.as_tensor(modes, dtype=torch.float64, device=DEV); B["MU_T"] = torch.as_tensor(mu, dtype=torch.float64, device=DEV); B["W_T"] = torch.as_tensor(B["W"], dtype=torch.float64, device=DEV)
    return B
def whitened(pts, lab, cnt, B):
    if USE_GPU:
        Fc = torch.zeros((len(cnt), M_MODES), dtype=torch.float64, device=DEV); Fs = torch.zeros_like(Fc)
        for s0 in range(0, len(pts), CHUNK):
            P = torch.as_tensor(pts[s0:s0 + CHUNK], dtype=torch.float64, device=DEV); ix = torch.as_tensor(lab[s0:s0 + CHUNK], dtype=torch.long, device=DEV)
            th = (2.0 * math.pi) * (P @ B["MT"].T); Fc.index_add_(0, ix, torch.cos(th)); Fs.index_add_(0, ix, torch.sin(th))
        c = torch.as_tensor(cnt, dtype=torch.float64, device=DEV)
        return (((torch.cat([Fc, Fs], 1) - c[:, None] * B["MU_T"][None, :]) / torch.sqrt(c)[:, None]) @ B["W_T"]).cpu().numpy()
    import scipy.sparse as sp
    X = np.zeros((len(cnt), 2 * M_MODES))
    for s0 in range(0, len(pts), CHUNK):
        P, l_ = pts[s0:s0 + CHUNK], lab[s0:s0 + CHUNK]; th = 2 * np.pi * (P @ B["modes"].T)
        X += sp.csr_matrix((np.ones(len(P)), (l_, np.arange(len(P)))), shape=(len(cnt), len(P))) @ np.hstack([np.cos(th), np.sin(th)])
    return ((X - cnt[:, None] * B["mu"][None, :]) / np.sqrt(cnt)[:, None]) @ B["W"]
def union_S(Y, cnt, ix): return float(((Y[ix] * np.sqrt(cnt[ix])[:, None]).sum(0) ** 2).mean() / cnt[ix].sum())
def mmd2(Xw, refs):
    # Liu-Guo-Dong style: unbiased MMD^2 between window sample and each benign reference batch, averaged
    def k(A_, B_):
        if USE_GPU: return torch.exp(-torch.cdist(A_, B_) ** 2 / (2 * MMD_BW ** 2))
        d2 = (A_ ** 2).sum(1)[:, None] + (B_ ** 2).sum(1)[None, :] - 2 * A_ @ B_.T; return np.exp(-np.maximum(d2, 0) / (2 * MMD_BW ** 2))
    X = torch.as_tensor(Xw, dtype=torch.float32, device=DEV) if USE_GPU else Xw
    n = len(X); Kxx = k(X, X); sxx = (Kxx.sum() - Kxx.diagonal().sum()) / (n * (n - 1)); out = []
    for R in refs:
        m = len(R); Kyy = k(R, R); Kxy = k(X, R)
        out.append(float(sxx + (Kyy.sum() - Kyy.diagonal().sum()) / (m * (m - 1)) - 2 * Kxy.mean()))
    return float(np.mean(out))
def fraudar(lab, top, A_):
    C = np.zeros((A_, N_TOPICS)); np.add.at(C, (lab, top), 1.0); wt = 1.0 / np.log(C.sum(0) + 5.0); Wm = C * wt[None, :]
    da, dt = Wm.sum(1), Wm.sum(0); alive_a, alive_t = np.ones(A_, bool), np.ones(N_TOPICS, bool); tot = Wm.sum(); n = A_ + N_TOPICS; best = tot / n
    for _ in range(n - 1):
        ia = np.where(alive_a, da, np.inf).argmin(); it = np.where(alive_t, dt, np.inf).argmin()
        if da[ia] <= dt[it]:
            alive_a[ia] = False; tot -= da[ia]; dt -= Wm[ia] * alive_t; da[ia] = np.inf
        else:
            alive_t[it] = False; tot -= dt[it]; da -= Wm[:, it] * alive_a; dt[it] = np.inf
        n -= 1; best = max(best, tot / max(n, 1))
    return float(best)
def window_stats(idx_pts, lab, cnt, Bc, Bf, refs, extra=None, en_extra=None, top_extra=None):
    pts = Z[idx_pts] if extra is None else np.vstack([extra[0], Z[idx_pts]]); labs = lab if extra is None else np.concatenate([extra[1], lab])
    en = EN[idx_pts] if extra is None else np.vstack([en_extra, EN[idx_pts]]); top = TOPIC[idx_pts] if extra is None else np.concatenate([top_extra, TOPIC[idx_pts]])
    Yc = whitened(pts, labs, cnt, Bc); Yf = whitened(pts, labs, cnt, Bf); A_ = len(cnt)
    ev, V = np.linalg.eigh(Yc.T @ Yc / Bc["r"]); v = Yc @ V[:, -1]; v2 = v ** 2 / (v ** 2).sum()
    o = np.argsort(-v2); m = int(np.clip(np.searchsorted(np.cumsum(v2[o]), 0.5) + 1, 3, 100)); grp = o[:m]
    r2 = (Yc ** 2).mean(1); rr = np.random.default_rng(len(labs)); samp = rr.choice(len(en), min(MMD_WIN, len(en)), replace=False)
    return dict(spec=float(ev[-1]), r2=r2, acct=float(r2.max()), glob=union_S(Yc, cnt, np.arange(A_)),
                mmd=mmd2(en[samp].astype(np.float32), refs), fraudar=fraudar(labs, top, A_),
                vol=float(np.sort(cnt)[-10:].sum() / cnt.sum()), twoscale=float(np.log(union_S(Yc, cnt, grp)) - np.log(union_S(Yf, cnt, grp))), v2=v2, grp=grp)
def hc_stat(x, ns):
    M_ = len(ns); p = np.sort((1 + M_ - np.searchsorted(ns, x, side="left")) / (M_ + 1)); A_ = len(p); k = max(1, A_ // 2)
    i = np.arange(1, k + 1); pp = p[:k]; return float(np.max(np.sqrt(A_) * (i / A_ - pp) / np.sqrt(pp * (1 - pp))))

In [ ]:
# ===== Cell 7: sources, query-cluster statistics, extended window statistics =====
FOLLOW = np.array([i for i in POOL_TXT if TURN[i] >= 2]) if len(POOL_TXT) else pool_idx
def _mm(key, ix): return DZ[key][ix], DIST[key][ix].astype(np.float32), km.predict(DZ[key][ix]).astype(np.int32)
def campaign_points(src, n, r):
    if src == "random": ix = r.choice(pool_idx, n, replace=False); return Z[ix], EN[ix].astype(np.float32), TOPIC[ix]
    if src == "wild_cluster": ix = r.choice(WILD, min(n, len(WILD)), replace=n > len(WILD)); return Z[ix], EN[ix].astype(np.float32), TOPIC[ix]
    if src in ("metamath_unique", "gsm8k_human", "metamath_template", "eval_gsm8k", "eval_mmlu"):
        D = DIST[src]; return _mm(src, r.choice(len(D), min(n, len(D)), replace=n > len(D)))
    if src == "metamath_camouflage": return _mm("metamath_unique", r.choice(CAMO, min(n, len(CAMO)), replace=n > len(CAMO)))
    if src == "metamath_rephrase":
        g = r.permutation(int(MM_GROUP.max()) + 1)[: int(math.ceil(n / 4))]; ix = np.flatnonzero(np.isin(MM_GROUP, g))[:n]; return _mm(src, ix)
    if src == "metamath_repeat":
        half = (n + 1) // 2; ix = r.choice(len(D_MMU), half, replace=False); ix = np.repeat(ix, 2)[:n]; return _mm("metamath_unique", ix)
    nm = (n + 1) // 2; ix = r.choice(len(D_MMU), nm, replace=False); other = FOLLOW if src == "metamath_followup" else pool_idx   # mixed / follow-up
    jx = r.choice(other, n - nm, replace=False); a, b, c = _mm("metamath_unique", ix)
    Zs, Es, Ts = np.empty((n, Z.shape[1]), np.float32), np.empty((n, EN.shape[1]), np.float32), np.empty(n, np.int32)
    Zs[0::2], Es[0::2], Ts[0::2] = a, b, c; Zs[1::2], Es[1::2], Ts[1::2] = Z[jx], EN[jx].astype(np.float32), TOPIC[jx]
    return Zs, Es, Ts                                                     # prompt and its padding/follow-up land in the same account

def cell_stats(pts, top, Bc, Bf):
    # two-scale score per topic cell, pooling queries across all accounts
    u, inv = np.unique(top, return_inverse=True); cnt = np.bincount(inv)
    yc = (whitened(pts, inv, cnt, Bc) ** 2).mean(1); yf = (whitened(pts, inv, cnt, Bf) ** 2).mean(1)
    ts = np.full(N_TOPICS, np.nan); n = np.zeros(N_TOPICS); n[u] = cnt; ok = cnt >= MINCELL
    ts[u[ok]] = np.log(yc[ok]) - np.log(yf[ok]); return ts, n / n.sum()
def window_stats15(idx_pts, lab, cnt, Bc, Bf, refs, extra=None, en_extra=None, top_extra=None):
    s = window_stats(idx_pts, lab, cnt, Bc, Bf, refs, extra=extra, en_extra=en_extra, top_extra=top_extra)
    pts = Z[idx_pts] if extra is None else np.vstack([extra[0], Z[idx_pts]]); labs = lab if extra is None else np.concatenate([extra[1], lab])
    en = EN[idx_pts] if extra is None else np.vstack([en_extra, EN[idx_pts]]); top = TOPIC[idx_pts] if extra is None else np.concatenate([top_extra, TOPIC[idx_pts]])
    s["cell_ts"], s["cell_share"] = cell_stats(pts, top, Bc, Bf)
    gm = np.isin(labs, s["grp"]); ge = en[gm]
    if len(ge) > MMD_WIN: ge = ge[np.random.default_rng(len(labs)).choice(len(ge), MMD_WIN, replace=False)]
    s["gmmd"] = mmd2(ge.astype(np.float32), refs) if len(ge) >= 20 else 0.0
    s["labs"] = labs; s["top"] = top; s["n_total"] = len(top); return s
def hc_stat(x, ns):
    M_ = len(ns); p = np.sort((1 + M_ - np.searchsorted(ns, x, side="left")) / (M_ + 1)); A_ = len(p); k = max(1, A_ // 2)
    i = np.arange(1, k + 1); pp = p[:k]; return float(np.max(np.sqrt(A_) * (i / A_ - pp) / np.sqrt(pp * (1 - pp))))
def calibrate15(recs):
    half = recs[0::2]; rest = recs[1::2] if len(recs) > 1 else recs
    TS = np.array([r["cell_ts"] for r in half]); SH = np.array([r["cell_share"] for r in half]); NT = np.array([r["n_total"] for r in half], float)
    LN = np.log(np.maximum(SH * NT[:, None], 1.0)); valid = np.mean(~np.isnan(TS), 0) >= 0.5
    A_ = np.zeros(N_TOPICS); B_ = np.zeros(N_TOPICS); SC = np.ones(N_TOPICS)
    for t in np.flatnonzero(valid):
        ok = ~np.isnan(TS[:, t]); x, y = LN[ok, t], TS[ok, t]
        b_, a_ = np.polyfit(x, y, 1) if np.ptp(x) > 0 else (0.0, float(np.median(y)))
        res = y - (a_ + b_ * x); A_[t], B_[t] = a_, b_; SC[t] = max(1.4826 * np.median(np.abs(res - np.median(res))), 0.05)
    med_sh = np.median(SH, 0); sc_sh = np.maximum(1.4826 * np.median(np.abs(SH - med_sh), 0), 0.002 + 0.1 * med_sh)
    T = dict(A=A_, B=B_, SC=SC, valid=valid, med_sh=med_sh, sc_sh=sc_sh)
    for r in recs: add_scores(r, T)
    q_ = lambda a: float(np.quantile(a, 1 - ALPHA))
    for d in DETS: T[d] = q_([r[d] for r in rest])
    return T
def add_scores(s, T):
    ln = np.log(np.maximum(s["cell_share"] * s["n_total"], 1.0))
    with np.errstate(all="ignore"):
        z = np.where(T["valid"] & ~np.isnan(s["cell_ts"]), (s["cell_ts"] - (T["A"] + T["B"] * ln)) / T["SC"], np.nan)
    s["qc"] = float(np.nanmax(z)) if np.any(~np.isnan(z)) else -1e9; s["qc_cell"] = int(np.nanargmax(z)) if np.any(~np.isnan(z)) else -1
    s["qcount"] = float(np.max((s["cell_share"] - T["med_sh"]) / T["sc_sh"])); return s
def fires15(s, T): return {d: s[d] > T[d] for d in DETS}
def boot_ci(x, B=1000, rng=np.random.default_rng(7)):
    x = np.array(x, float); bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]; return float(x.mean()), float(np.quantile(bs, .025)), float(np.quantile(bs, .975))
def roc(null_vals, att_vals):
    null_vals, att_vals = np.array(null_vals), np.array(att_vals)
    auc = float(np.mean(att_vals[:, None] > null_vals[None, :]) + 0.5 * np.mean(att_vals[:, None] == null_vals[None, :]))
    return dict(auc=auc, tpr1=float(np.mean(att_vals > np.quantile(null_vals, 0.99))), tpr5=float(np.mean(att_vals > np.quantile(null_vals, 0.95))))

In [ ]:
# ===== Cell 8: sparse-support statistics on the whitened account rows =====
import time as _time
def rows(pts, lab, cnt, B): return whitened(pts, lab, cnt, B)
def spectral(Y):
    t = _time.perf_counter(); ev, V = np.linalg.eigh(Y.T @ Y / Y.shape[1]); return float(ev[-1]), V[:, -1], _time.perf_counter() - t
def tpower(Y, k, iters, restarts, v_top, rng):
    t = _time.perf_counter(); A_, r = Y.shape; k = int(min(k, A_)); best, best_s = -1.0, None
    u = Y @ v_top; inits = [np.argpartition(-np.abs(u), k - 1)[:k]] + [rng.choice(A_, k, replace=False) for _ in range(restarts - 1)]
    for s0 in inits:
        z = np.zeros(A_); z[s0] = 1 / np.sqrt(k)
        for _ in range(iters):
            w = Y @ (Y.T @ z) / r; s_ = np.argpartition(-np.abs(w), k - 1)[:k]; z = np.zeros(A_); z[s_] = w[s_]; z /= (np.linalg.norm(z) + 1e-12)
        val = float(np.sum((Y.T @ z) ** 2) / r)
        if val > best: best, best_s = val, np.flatnonzero(z)
    return best, best_s, _time.perf_counter() - t
def greedy_scan(Y, k):
    t = _time.perf_counter(); A_, r = Y.shape; k = int(min(k, A_)); S = np.zeros(r); used = np.zeros(A_, bool); best = -1.0; sel = []
    for j in range(1, k + 1):
        sc = ((S[None, :] + Y) ** 2).sum(1) / (r * j); sc[used] = -np.inf; i = int(np.argmax(sc)); used[i] = True; sel.append(i); S += Y[i]
        best = max(best, float(sc[i]))
    return best, np.array(sel), _time.perf_counter() - t
TP_CFG = [(5, 1), (20, 5), (100, 20)]
def sparse_suite(pts, lab, cnt, Bc, K_, rng, colluders=None):
    Y = rows(pts, lab, cnt, Bc); lm, v, t_sp = spectral(Y); out = dict(spectral=lm, t_spectral=t_sp)
    for kk in ["K", "2K"]:
        k = K_ if kk == "K" else 2 * K_
        for it, rs in TP_CFG:
            val, sup, tt = tpower(Y, k, it, rs, v, rng); out[f"tp_{kk}_{it}x{rs}"] = val; out[f"t_tp_{kk}_{it}x{rs}"] = tt
        val, sup, tt = greedy_scan(Y, k); out[f"greedy_{kk}"] = val; out[f"t_greedy_{kk}"] = tt
    sq = np.sqrt(cnt)
    if colluders is None: colluders = rng.choice(len(cnt), K_, replace=False)                     # null: a random K-account union
    out["oracle"] = float(((Y[colluders] * sq[colluders, None]).sum(0) ** 2).mean() / cnt[colluders].sum())
    return out
SPARSE_KEYS = None

## Part S — sparse-support detection vs compute vs K/A

In [ ]:
# ===== Cell 8: weekly populations — calibration, paired detection, ROC at matched realised FPR, cumulative detection =====
t0 = time.time(); blk = WEEK.astype(np.int32); n_b = int(blk.max()) + 1
order = np.lexsort((ACCT, blk)); keys = blk[order].astype(np.int64) * n_acct + ACCT[order]; GA = {b: [] for b in range(n_b)}
for g in np.split(order, np.flatnonzero(np.diff(keys)) + 1):
    if not is_pool[int(ACCT[g[0]])] and len(g) >= MIN_PER_ACCT: GA[int(blk[g[0]])].append(g[:VOL_CAP])
ok = [b for b in range(WK["CAL_N"], n_b) if len(GA[b]) >= 40 and all(len(GA[v]) >= 20 for v in range(b - WK["CAL_N"], b))]
TB = [int(ok[i]) for i in np.linspace(0, len(ok) - 1, min(WK["N_TEST"], len(ok))).astype(int)]
def sub_window(b, r):
    k = max(20, int(SUB_FRAC * len(GA[b]))); pick = r.choice(len(GA[b]), k, replace=False); parts = [GA[b][i] for i in pick]
    vols = np.array([len(p) for p in parts]); return np.concatenate(parts), np.repeat(np.arange(len(parts)), vols), vols
NEED = sorted(set(v for b in TB for v in range(b - WK["CAL_N"], b + 1))); BANK, REFS, NULLW = {}, {}, {}
for b in NEED:
    prev = [np.concatenate(GA[v]) for v in range(max(0, b - WK["LAGB"]), b) if GA[v]]
    ref = np.concatenate(prev) if prev else np.concatenate(GA[b]); ref = np.random.default_rng(b).choice(ref, min(100000, len(ref)), replace=False)
    BANK[b] = (make_bank_aug(Z[ref], H_COARSE, SEED + 7), make_bank_aug(Z[ref], H_FINE, SEED + 8))
    rb = np.random.default_rng(b + 1).choice(ref, N_REF_BATCH * REF_BATCH, replace=len(ref) < N_REF_BATCH * REF_BATCH).reshape(N_REF_BATCH, REF_BATCH)
    REFS[b] = [torch.as_tensor(EN[x].astype(np.float32), device=DEV) if USE_GPU else EN[x].astype(np.float32) for x in rb]
    NULLW[b] = [window_stats15(*sub_window(b, np.random.default_rng(SEED + 100 * b + t))[:2], np.bincount(sub_window(b, np.random.default_rng(SEED + 100 * b + t))[1]), *BANK[b], REFS[b]) for t in range(WK["NULL_PER"])]
print(f"[weekly] test weeks {TB}; banks and null windows for {len(NEED)} weeks ({time.time()-t0:.0f}s)")
THR = {b: calibrate15([r for v in range(b - WK["CAL_N"], b) for r in NULLW[v]]) for b in TB}
for b in TB:
    for r in NULLW[b]: add_scores(r, THR[b])
# ===== Part S: sparse-support detection vs compute vs K/A =====
t0 = time.time(); RS = []
def inject(idx, lab, src, K_, N, b_or_vols, r, schedule="random", lam=None, bank_rows=None):
    vols = np.array([len(g) for g in GA[b_or_vols]]) if isinstance(b_or_vols, (int, np.integer)) else b_or_vols
    cv = r.choice(vols, K_, replace=True).astype(float); cv = np.maximum(1, np.round(cv * N / cv.sum())).astype(int); N = int(cv.sum())
    if lam is not None:
        nm = int(round((1 - lam) * N)); Zm, Em, Tm = campaign_points("metamath_unique", nm, r) if nm > 0 else (np.empty((0, Z.shape[1]), np.float32), np.empty((0, EN.shape[1]), np.float32), np.empty(0, np.int32))
        jx = r.choice(pool_idx, N - nm, replace=False); Zc = np.vstack([Zm, Z[jx]]); Ec = np.vstack([Em, EN[jx].astype(np.float32)]); Tc = np.concatenate([Tm, TOPIC[jx]])
        p = r.permutation(len(Zc)); Zc, Ec, Tc = Zc[p], Ec[p], Tc[p]
    else:
        Zc, Ec, Tc = campaign_points(src, N, r)
    N = len(Zc); labc = np.repeat(np.arange(K_), cv)[:N]
    if schedule != "random" and bank_rows is not None:
        proj = bank_rows(Zc)                                                    # distinctiveness of each prompt under the defender-like bank
        o = np.argsort(-proj)
        if schedule == "balanced": labc = np.empty(N, int); labc[o] = np.arange(N) % K_
        elif schedule == "sacrificial":
            n_top = int(0.3 * N); labc = np.empty(N, int); labc[o[:n_top]] = np.arange(n_top) % 5; labc[o[n_top:]] = 5 + np.arange(N - n_top) % (K_ - 5)
    keep = lab >= K_; _, inv = np.unique(np.concatenate([labc, lab[keep]]), return_inverse=True)
    return np.vstack([Zc, Z[idx[keep]]]), inv, np.bincount(inv), Zc, Ec, Tc, inv[:N]
# weekly windows, campaign fixed at N_FIX over K colluders
NULL_S = []
for b in TB:
    for rep in range(2):
        r = np.random.default_rng(SEED + 5_000 * b + rep); idx, lab, vols = sub_window(b, r)
        for K_ in P21["KS"]:
            if K_ >= (lab.max() + 1) // 2: continue
            NULL_S.append(dict(where="weekly", K=K_, **sparse_suite(Z[idx], lab, np.bincount(lab), BANK[b][0], K_, r)))
for K_ in P21["KS"]:
    for src in ["metamath_unique", "metamath_mixed"]:
        for b in TB:
            for rep in range(2):
                r = np.random.default_rng(SEED + 6_000 * b + 10 * rep + K_); idx, lab, vols = sub_window(b, r); A_ = int(lab.max()) + 1
                if K_ >= A_ // 2: continue
                pts, labs, cnt, *_ , lc = inject(idx, lab, src, K_, P21["N_FIX"], b, r)
                RS.append(dict(where="weekly", K=K_, A=A_, source=src, **sparse_suite(pts, labs, cnt, BANK[b][0], K_, r, colluders=np.arange(K_))))
print(f"[S] weekly done ({time.time()-t0:.0f}s)")
# pooled populations of account-weeks (A = 1,000), one reference, oracle thresholds only
order5 = np.lexsort((ACCT, WEEK)); keys5 = WEEK[order5].astype(np.int64) * n_acct + ACCT[order5]
POOLW = [g[:VOL_CAP] for g in np.split(order5, np.flatnonzero(np.diff(keys5)) + 1) if not is_pool[int(ACCT[g[0]])] and len(g) >= MIN_PER_ACCT]
refP = np.random.default_rng(12).choice(np.concatenate(POOLW), min(100000, sum(len(g) for g in POOLW)), replace=False)
BP = (make_bank_aug(Z[refP], H_COARSE, SEED + 7), make_bank_aug(Z[refP], H_FINE, SEED + 8))
rbp = np.random.default_rng(13).choice(refP, N_REF_BATCH * REF_BATCH, replace=len(refP) < N_REF_BATCH * REF_BATCH).reshape(N_REF_BATCH, REF_BATCH)
REFP_ = [torch.as_tensor(EN[x].astype(np.float32), device=DEV) if USE_GPU else EN[x].astype(np.float32) for x in rbp]
def pool_window(A_, r):
    pick = r.choice(len(POOLW), min(A_, len(POOLW)), replace=False); parts = [POOLW[i] for i in pick]
    vols = np.array([len(p) for p in parts]); return np.concatenate(parts), np.repeat(np.arange(len(parts)), vols), vols
AP = min(P21["A_POOL"], int(0.8 * len(POOLW)))
for t in range(P21["NULL_POOL"]):
    r = np.random.default_rng(SEED + 7_000 + t); idx, lab, vols = pool_window(AP, r)
    for K_ in P21["KS"]: NULL_S.append(dict(where="pooled", K=K_, **sparse_suite(Z[idx], lab, vols, BP[0], K_, r)))
for K_ in P21["KS"]:
    for src in ["metamath_unique", "metamath_mixed"]:
        for t in range(P21["ATT_POOL"]):
            r = np.random.default_rng(SEED + 8_000 + 31 * t + K_); idx, lab, vols = pool_window(AP, r)
            pts, labs, cnt, *_ , lc = inject(idx, lab, src, K_, P21["N_FIX"], vols, r)
            RS.append(dict(where="pooled", K=K_, A=AP, source=src, **sparse_suite(pts, labs, cnt, BP[0], K_, r, colluders=np.arange(K_))))
print(f"[S] pooled A={AP} done ({time.time()-t0:.0f}s)")
STATS = [k for k in RS[0] if not k.startswith("t_") and k not in ("where", "K", "A", "source")]
SPARSE_ROWS = []
for where in ["weekly", "pooled"]:
    for K_ in P21["KS"]:
        nulls = [n for n in NULL_S if n["where"] == where and n["K"] == K_]
        if not nulls: continue
        for src in ["metamath_unique", "metamath_mixed"]:
            att = [a for a in RS if a["where"] == where and a["K"] == K_ and a["source"] == src]
            if not att: continue
            row = dict(where=where, K=K_, KA=float(np.mean([K_ / a["A"] for a in att])), source=src)
            for st in STATS:
                thr = np.quantile([n[st] for n in nulls], 0.99); row[st] = float(np.mean([a[st] > thr for a in att]))
                tk = "t_" + st; row[tk] = float(np.mean([a[tk] for a in att])) if tk in att[0] else float(np.mean([a["t_spectral"] for a in att]))
            SPARSE_ROWS.append(row)
            # [PAPER] App. F 'Scale': sparse scans caught 0–4% at colluder fractions 1% to 28%, oracle told the colluders caught all; App. P (P21.1-P21.3)
            print(f"[S] {where:>6} K={K_:3d} (K/A {100*row['KA']:.1f}%) {src:>16}: spectral {row['spectral']:.2f} | oracle {row['oracle']:.2f} | "
                  + " ".join(f"{st}={row[st]:.2f}" for st in STATS if st.startswith(("tp_", "greedy"))) + f" | t_spec {1e3*row['t_spectral']:.1f}ms t_tp100x20(K) {1e3*row.get('t_tp_K_100x20', 0):.0f}ms")

## Part B — empirical detection boundary

In [ ]:
# ===== Part B: empirical detection boundary along P_lambda = (1-lambda) MetaMathQA + lambda natural =====
t0 = time.time(); REFX = [x for x in REFS[TB[0]]]
def mmd_eps(Ec): return mmd2(Ec[np.random.default_rng(0).choice(len(Ec), min(1000, len(Ec)), replace=False)].astype(np.float32), REFX)
BOUND = []
for where in ["weekly", "pooled"]:
    for lam in P21["LAMBDAS"]:
        vals_null, vals_att, eps = [], [], []
        for t in range(24 if where == "weekly" else 12):
            if where == "weekly":
                b = TB[t % len(TB)]; r = np.random.default_rng(SEED + 9_100 + t + int(100 * lam)); idx, lab, vols = sub_window(b, r); Bc, Bf, refs = BANK[b][0], BANK[b][1], REFS[b]
                if 50 >= (lab.max() + 1) // 2: continue
                bv = b
            else:
                r = np.random.default_rng(SEED + 9_200 + t + int(100 * lam)); idx, lab, vols = pool_window(AP, r); Bc, Bf, refs = BP[0], BP[1], REFP_; bv = vols
            sb = window_stats15(idx, lab, np.bincount(lab), Bc, Bf, refs); vals_null.append(sb["twoscale"])
            pts, labs, cnt, Zc, Ec, Tc, lc = inject(idx, lab, None, 50, P21["N_FIX"], bv, r, lam=lam)
            sa = window_stats15(idx[lab >= 50], labs[len(Zc):], cnt, Bc, Bf, refs, extra=(Zc, lc), en_extra=Ec, top_extra=Tc); vals_att.append(sa["twoscale"]); eps.append(mmd_eps(Ec))
        BOUND.append(dict(where=where, lam=lam, eps=float(np.mean(eps)), null=vals_null, att=vals_att))
    print(f"[B] {where} done ({time.time()-t0:.0f}s)")
# benign groups' law gap: MMD of the top-eigenvector group's queries in null windows
GEPS = {}
for where in ["weekly", "pooled"]:
    ge = []
    for t in range(40 if where == "weekly" else 20):
        if where == "weekly":
            b = TB[t % len(TB)]; r = np.random.default_rng(SEED + 9_300 + t); idx, lab, vols = sub_window(b, r); s = window_stats15(idx, lab, np.bincount(lab), BANK[b][0], BANK[b][1], REFS[b])
        else:
            r = np.random.default_rng(SEED + 9_400 + t); idx, lab, vols = pool_window(AP, r); s = window_stats15(idx, lab, np.bincount(lab), BP[0], BP[1], REFP_)
        gm = np.isin(lab, s["grp"]); ge.append(mmd_eps(EN[idx][gm].astype(np.float32)) if gm.sum() >= 20 else 0.0)
    GEPS[where] = ge
for where in ["weekly", "pooled"]:
    rows_ = [x for x in BOUND if x["where"] == where]; allnull = np.concatenate([x["null"] for x in rows_]); thr = np.quantile(allnull, 0.99)
    for x in rows_: x["tpr"] = float(np.mean(np.array(x["att"]) > thr))
    es, tp = [x["eps"] for x in rows_], [x["tpr"] for x in rows_]; eps_star = float("nan")
    order_ = np.argsort(es); es, tp = np.array(es)[order_], np.array(tp)[order_]
    for i in range(len(es) - 1):
        if tp[i] < 0.5 <= tp[i + 1]: eps_star = float(es[i] + (0.5 - tp[i]) / (tp[i + 1] - tp[i]) * (es[i + 1] - es[i])); break
    # [PAPER] App. F 'Statistics that failed': 0.81 (f = 0) to 0.00 (f = 0.75), crossing 0.5 at MMD2 about 0.035; benign top groups median 0.145
    print(f"[B] {where}: " + " ".join(f"lam={x['lam']}: eps={x['eps']:.4f} TPR={x['tpr']:.2f}" for x in rows_) + f" | eps*={eps_star:.4f} | benign top-group eps p50/p99 = {np.median(GEPS[where]):.4f}/{np.quantile(GEPS[where], .99):.4f}")
    for x in rows_: x["eps_star"] = eps_star

## Part X — adversarial scheduling

In [ ]:
# ===== Part X: cross-account adversarial scheduling =====
t0 = time.time(); SCHED = {}
def proj_fn(b):
    Bc = BANK[b][0]
    def f(Zc):
        Y1 = whitened(Zc, np.arange(len(Zc)), np.ones(len(Zc)), Bc); m = Y1.mean(0); return Y1 @ m / (np.linalg.norm(m) + 1e-12)
    return f
for sched in ["random", "balanced", "sacrificial"]:
    det, prec, rec_nonsac, rec_all = [], [], [], []; nv, av = [], []
    for b in TB:
        for rep in range(3):
            r = np.random.default_rng(SEED + 9_500 + 1000 * b + rep); idx, lab, vols = sub_window(b, r)
            if 50 >= (lab.max() + 1) // 2: continue
            sb = window_stats15(idx, lab, np.bincount(lab), *BANK[b], REFS[b]); thr = THR[b]["twoscale"]
            pts, labs, cnt, Zc, Ec, Tc, lc = inject(idx, lab, "metamath_unique", 50, P21["N_FIX"], b, r, schedule=sched, bank_rows=proj_fn(b))
            sa = window_stats15(idx[lab >= 50], labs[len(Zc):], cnt, *BANK[b], REFS[b], extra=(Zc, lc), en_extra=Ec, top_extra=Tc)
            f = sa["twoscale"] > thr and not (sb["twoscale"] > thr); det.append(float(f)); nv.append(sb["twoscale"]); av.append(sa["twoscale"])
            if f:
                g = sa["grp"]; prec.append(float(np.mean(g < 50))); rec_all.append(float(np.sum(g < 50) / 50))
                rec_nonsac.append(float(np.sum((g >= 5) & (g < 50)) / 45) if sched == "sacrificial" else float("nan"))
    SCHED[sched] = dict(paired=float(np.mean(det)), precision=float(np.nanmean(prec)) if prec else float("nan"), recall=float(np.nanmean(rec_all)) if rec_all else float("nan"),
                        recall_nonsac=float(np.nanmean(rec_nonsac)) if rec_nonsac and not np.all(np.isnan(rec_nonsac)) else float("nan"), oracle=roc(nv, av)["tpr1"])
    # [PAPER] §5 'Attacks on the monitor' / App. D.6: sacrificial detection on C 0.92, none of the other 45 colluders flagged
    print(f"[X] {sched:>11}: two-scale paired {SCHED[sched]['paired']:.2f} (oracle {SCHED[sched]['oracle']:.2f}) | flagged precision {SCHED[sched]['precision']:.2f}, recall {SCHED[sched]['recall']:.2f}, non-sacrificial recall {SCHED[sched]['recall_nonsac']:.2f}  ({time.time()-t0:.0f}s)")

## Part T — temporal camouflage

In [ ]:
# ===== Part T: temporal camouflage — a campaign ramping up through the calibration weeks =====
t0 = time.time(); TEMP = {"abrupt": [], "ramped": []}
TWK = [b for b in TB if b - WK["CAL_N"] - WK["LAGB"] >= 0][:: max(1, len(TB) // P21["T_WEEKS"])][: P21["T_WEEKS"]]
for b in TWK:
    # contaminated lagged reference and calibration windows: in week b-j the campaign runs at volume (1 - j/8) of full
    camp_ref = campaign_points("metamath_unique", P21["N_FIX"] * 2, np.random.default_rng(b))[0]
    prev = [np.concatenate(GA[v]) for v in range(b - WK["LAGB"], b) if GA[v]]; ref = np.random.default_rng(b).choice(np.concatenate(prev), 100000 if not QUICK else 5000, replace=True)
    frac_ref = np.mean([1 - j / WK["CAL_N"] for j in range(1, WK["LAGB"] + 1)]); n_c = int(frac_ref * P21["N_FIX"] * WK["LAGB"] / (sum(len(np.concatenate(GA[v])) for v in range(b - WK["LAGB"], b)) / len(ref)))
    Zref_c = np.vstack([Z[ref], camp_ref[: max(0, min(n_c, len(camp_ref)))]])
    Bt = (make_bank_aug(Zref_c, H_COARSE, SEED + 7), make_bank_aug(Zref_c, H_FINE, SEED + 8))
    cal = []
    for j in range(1, WK["CAL_N"] + 1):
        v = b - j; vol = max(1, int((1 - j / WK["CAL_N"]) * P21["N_FIX"]))
        for k in range(P21["T_CAL"]):
            r = np.random.default_rng(SEED + 9_700 + 100 * v + k); idx, lab, vols = sub_window(v, r)
            if 50 >= (lab.max() + 1) // 2 or vol < 50: cal.append(window_stats15(idx, lab, np.bincount(lab), *Bt, REFS[b])); continue
            pts, labs, cnt, Zc, Ec, Tc, lc = inject(idx, lab, "metamath_unique", 50, vol, v, r)
            cal.append(window_stats15(idx[lab >= 50], labs[len(Zc):], cnt, *Bt, REFS[b], extra=(Zc, lc), en_extra=Ec, top_extra=Tc))
    thr_ramp = float(np.quantile([c["twoscale"] for c in cal], 1 - ALPHA)); thr_abr = THR[b]["twoscale"]
    for rep in range(P21["T_REPS"]):
        r = np.random.default_rng(SEED + 9_800 + 100 * b + rep); idx, lab, vols = sub_window(b, r)
        if 50 >= (lab.max() + 1) // 2: continue
        pts, labs, cnt, Zc, Ec, Tc, lc = inject(idx, lab, "metamath_unique", 50, P21["N_FIX"], b, r)
        sb_a = window_stats15(idx, lab, np.bincount(lab), *BANK[b], REFS[b]); sa_a = window_stats15(idx[lab >= 50], labs[len(Zc):], cnt, *BANK[b], REFS[b], extra=(Zc, lc), en_extra=Ec, top_extra=Tc)
        sb_r = window_stats15(idx, lab, np.bincount(lab), *Bt, REFS[b]); sa_r = window_stats15(idx[lab >= 50], labs[len(Zc):], cnt, *Bt, REFS[b], extra=(Zc, lc), en_extra=Ec, top_extra=Tc)
        TEMP["abrupt"].append(float(sa_a["twoscale"] > thr_abr and not sb_a["twoscale"] > thr_abr)); TEMP["ramped"].append(float(sa_r["twoscale"] > thr_ramp and not sb_r["twoscale"] > thr_ramp))
    print(f"[T] week {b}: abrupt {np.mean(TEMP['abrupt']):.2f} vs ramped {np.mean(TEMP['ramped']):.2f} (running)  ({time.time()-t0:.0f}s)")
# [PAPER] §5 'Attacks on the monitor' / App. D.6: ramp cuts detection from 0.95 (abrupt) to 0.12 (ramped)
print(f"[T] deployed two-scale paired detection: abrupt {np.mean(TEMP['abrupt']):.2f}, ramped through calibration {np.mean(TEMP['ramped']):.2f}")

## Part R — low-FPR ROC

In [ ]:
# ===== Part R: low-FPR ROC (oracle) with many null windows =====
t0 = time.time(); RN = {d: [] for d in ["twoscale", "glob", "mmd"]}; RA = {(s_, d): [] for s_ in ["metamath_unique", "metamath_mixed"] for d in ["twoscale", "glob", "mmd"]}
for b in TB:
    for r_ in NULLW[b]:
        for d in RN: RN[d].append(r_[d])
    for k in range(P21["R_EXTRA"]):
        r = np.random.default_rng(SEED + 10_000 + 1000 * b + k); idx, lab, vols = sub_window(b, r); s = window_stats15(idx, lab, np.bincount(lab), *BANK[b], REFS[b])
        for d in RN: RN[d].append(s[d])
    for s_ in ["metamath_unique", "metamath_mixed"]:
        for rep in range(4):
            r = np.random.default_rng(SEED + 11_000 + 1000 * b + rep); idx, lab, vols = sub_window(b, r)
            if 50 >= (lab.max() + 1) // 2: continue
            pts, labs, cnt, Zc, Ec, Tc, lc = inject(idx, lab, s_, 50, P21["N_FIX"], b, r)
            sa = window_stats15(idx[lab >= 50], labs[len(Zc):], cnt, *BANK[b], REFS[b], extra=(Zc, lc), en_extra=Ec, top_extra=Tc)
            for d in ["twoscale", "glob", "mmd"]: RA[(s_, d)].append(sa[d])
LOWFPR = {}
for (s_, d), v in RA.items():
    nul = np.array(RN[d]); LOWFPR[(s_, d)] = {f: float(np.mean(np.array(v) > np.quantile(nul, 1 - f))) for f in [1e-1, 1e-2, 1e-3]}
    # [PAPER] App. D.2: TPR at 10−3 two-scale 0.88, global stream 0.70, MMD 0.39 (metamath_unique rows)
    print(f"[R] {s_:>16} {d:>8}: TPR at FPR 1e-1/1e-2/1e-3 = " + "/".join(f"{LOWFPR[(s_, d)][f]:.2f}" for f in [1e-1, 1e-2, 1e-3]) + f"  (null windows {len(nul)})")
print(f"[R] ({time.time()-t0:.0f}s)")

## Predictions, figure, VERDICT REPORT — copy into the chat

In [ ]:
# ===== Cell 12: predictions, figure, verdict =====
pool_rows = [r for r in SPARSE_ROWS if r["where"] == "pooled" and r["source"] == "metamath_unique"]
# [PAPER] App. P (Step 21, P21.1/P21.2): Fail (0.03 vs 0.03); Pass (1.00; random-group null)
for r in pool_rows:
    best = max([k for k in r if k.startswith(("tp_", "greedy")) and not k.startswith("t_")], key=lambda k: r[k])
    print(f"P21.1/2 pooled K/A={100*r['KA']:.1f}%: spectral {r['spectral']:.2f}, best sparse {best} {r[best]:.2f} (compute {r['t_'+best]/max(r['t_spectral'],1e-9):.2f}x spectral), oracle-told {r['oracle']:.2f}")
# [PAPER] App. P (Step 21, P21.3): Pass (uninformative: both ≈ 0)
for r in [r for r in SPARSE_ROWS if r["where"] == "weekly" and r["source"] == "metamath_unique"]:
    best = max([k for k in r if k.startswith(("tp_", "greedy")) and not k.startswith("t_")], key=lambda k: r[k])
    print(f"P21.3 weekly K/A={100*r['KA']:.1f}%: spectral {r['spectral']:.2f} vs best sparse {r[best]:.2f}")
# [PAPER] App. P (Step 21, P21.4): Fail (0.035 vs 0.447)
for where in ["weekly", "pooled"]:
    x = [b for b in BOUND if b["where"] == where]
    if x: print(f"P21.4 {where}: eps* {x[0]['eps_star']:.4f} vs benign top-group eps p99 {np.quantile(GEPS[where], .99):.4f}")
# [PAPER] App. P (Step 21, P21.5): Pass (0.92; 0%)
print(f"P21.5 sacrificial: two-scale paired {SCHED['sacrificial']['paired']:.2f}, non-sacrificial recall {SCHED['sacrificial']['recall_nonsac']:.2f} (vs random: {SCHED['random']['paired']:.2f}, recall {SCHED['random']['recall']:.2f})")
# [PAPER] App. P (Step 21, P21.6): Pass (0.95 to 0.12)
print(f"P21.6 temporal camouflage: abrupt {np.mean(TEMP['abrupt']):.2f} vs ramped {np.mean(TEMP['ramped']):.2f} (pass if drop >= 0.3)")
# [PAPER] App. P (Step 21, P21.7): Pass (0.88 vs 0.70)
print(f"P21.7 TPR at FPR 1e-3 (unique): two-scale {LOWFPR[('metamath_unique','twoscale')][1e-3]:.2f}, global {LOWFPR[('metamath_unique','glob')][1e-3]:.2f}, MMD {LOWFPR[('metamath_unique','mmd')][1e-3]:.2f}")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for where, mk in [("weekly", "o"), ("pooled", "s")]:
    rr = sorted([r for r in SPARSE_ROWS if r["where"] == where and r["source"] == "metamath_unique"], key=lambda r: r["KA"])
    for st, c_ in [("spectral", "tab:blue"), ("tp_K_100x20", "tab:orange"), ("greedy_K", "tab:green"), ("oracle", "k")]:
        if rr: ax[0].plot([100 * r["KA"] for r in rr], [r[st] for r in rr], mk + "-", color=c_, label=f"{st} ({where})")
ax[0].set_xscale("log"); ax[0].set_xlabel("colluder fraction K/A (%)"); ax[0].set_ylabel("oracle TPR at 1%"); ax[0].legend(fontsize=6); ax[0].set_title("oracle → sparse scan → spectral")
for where, c_ in [("weekly", "tab:blue"), ("pooled", "tab:red")]:
    x = [b for b in BOUND if b["where"] == where]; ax[1].plot([b["eps"] for b in x], [b["tpr"] for b in x], "o-", color=c_, label=where)
    ax[1].axvline(np.quantile(GEPS[where], .99), color=c_, ls=":")
ax[1].set_xlabel("law gap ε (MMD² to benign reference)"); ax[1].set_ylabel("two-scale oracle TPR at 1%"); ax[1].legend(); ax[1].set_title("empirical detection boundary (dotted: benign top-group ε p99)")
for (s_, d), v in LOWFPR.items(): ax[2].plot([1e-1, 1e-2, 1e-3], [v[f] for f in [1e-1, 1e-2, 1e-3]], "o-", label=f"{s_} {d}")
ax[2].set_xscale("log"); ax[2].invert_xaxis(); ax[2].set_xlabel("oracle false-alarm rate"); ax[2].set_ylabel("TPR"); ax[2].legend(fontsize=6); ax[2].set_title("low-FPR operating points")
plt.tight_layout(); plt.savefig(f"{OUTDIR}/step21_figure.png", dpi=130); plt.show()
print("=" * 90); print("STEP 21 VERDICT REPORT"); print("=" * 90)
for r in SPARSE_ROWS: print("[S]", json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}))
for b in BOUND: print(f"[B] {b['where']} lam={b['lam']} eps={b['eps']:.5f} tpr={b['tpr']:.2f} eps*={b['eps_star']:.5f}")
for w_, g_ in GEPS.items(): print(f"[B] benign top-group eps {w_}: p50 {np.median(g_):.5f} p90 {np.quantile(g_, .9):.5f} p99 {np.quantile(g_, .99):.5f}")
for k, v in SCHED.items(): print("[X]", k, json.dumps({kk: round(vv, 3) for kk, vv in v.items()}))
print(f"[T] abrupt {np.mean(TEMP['abrupt']):.3f} (n={len(TEMP['abrupt'])}), ramped {np.mean(TEMP['ramped']):.3f}")
for k, v in LOWFPR.items(): print("[R]", k, json.dumps({str(f): round(x, 3) for f, x in v.items()}))
print("=" * 90)
with open(f"{OUTDIR}/step21_results.json", "w") as fh: json.dump(dict(S=SPARSE_ROWS, B=[{k: v for k, v in b.items() if k not in ("null", "att")} for b in BOUND], X=SCHED, T={k: float(np.mean(v)) for k, v in TEMP.items()}, R={str(k): v for k, v in LOWFPR.items()}), fh, indent=1, default=float)